In [2]:
# strict_er_cifar100_output_format.py

import os
import json
import random
from datetime import datetime
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import transforms
from torchvision.datasets import CIFAR100


# =========================
# CONFIG
# =========================
SAVE_PATH = "strict_er_cifar100_results.json"

QUICK_RUN = False
SEEDS = [1, 2, 3]

EPOCHS_A = 3 if QUICK_RUN else 12
EPOCHS_B = 3 if QUICK_RUN else 12

REPLAY_PER_CLASS = 40 if QUICK_RUN else 150
EVAL_TEMP = 1.0

BS = 128
NW = 4
LR = 1e-3
WD = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# UTILS
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_progress(meta: dict):
    with open(SAVE_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)


def build_memory_indices(train_targets, train_A_idx, replay_per_class, seed):
    rng = np.random.default_rng(seed)
    targets = np.array(train_targets)
    A_idx = np.array(train_A_idx)

    mem = []
    for c in sorted(np.unique(targets[A_idx])):
        cls_idx = A_idx[targets[A_idx] == c]
        k = min(replay_per_class, len(cls_idx))
        chosen = rng.choice(cls_idx, size=k, replace=False)
        mem.extend(chosen.tolist())

    rng.shuffle(mem)
    return mem


def stratified_split(indices, targets, val_ratio=0.2, test_ratio=0.2, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.array(indices)
    y = np.array(targets)

    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(y[idx]):
        c_idx = idx[y[idx] == c]
        rng.shuffle(c_idx)

        n = len(c_idx)
        n_test = int(round(n * test_ratio))
        n_val = int(round(n * val_ratio))
        n_train = n - n_val - n_test

        train_idx.extend(c_idx[:n_train].tolist())
        val_idx.extend(c_idx[n_train:n_train + n_val].tolist())
        test_idx.extend(c_idx[n_train + n_val:].tolist())

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


# =========================
# MODEL
# =========================
class SmallCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32->16
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 16->8
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),# 8->4
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# =========================
# TRAIN / EVAL
# =========================
def train_epochs(model, loader, epochs, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()


@torch.no_grad()
def eval_taskaware(model, loader, allowed_classes, device):
    model.eval()
    allowed = torch.tensor(allowed_classes, dtype=torch.long, device=device)
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        restricted = logits[:, allowed]
        pred_local = restricted.argmax(dim=1)
        pred = allowed[pred_local]
        correct += (pred == y).sum().item()
        total += y.size(0)

    return 100.0 * correct / max(total, 1)


@torch.no_grad()
def eval_cil(model, loader, device, temp=1.0):
    model.eval()
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x) / temp
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return 100.0 * correct / max(total, 1)


def run_er_cifar100_strict(
    seed: int,
    loaders: Tuple[DataLoader, DataLoader, DataLoader, DataLoader, DataLoader, DataLoader],
    device,
    train_base,
    memory_indices
):
    set_seed(seed)

    train_A_loader, val_A_loader, train_B_loader, val_B_loader, test_A_loader, test_B_loader = loaders

    model = SmallCNN(num_classes=100).to(device)

    # Phase A
    train_epochs(model, train_A_loader, EPOCHS_A, device)
    acc_A_init = eval_taskaware(model, test_A_loader, allowed_classes=list(range(0, 50)), device=device)

    # Phase B + replay memory from A
    mem_loader = DataLoader(
        Subset(train_base, memory_indices),
        batch_size=BS,
        shuffle=True,
        num_workers=NW,
        pin_memory=True,
        persistent_workers=(NW > 0)
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

    for _ in range(EPOCHS_B):
        model.train()
        mem_iter = iter(mem_loader)
        for xb, yb in train_B_loader:
            xb, yb = xb.to(device), yb.to(device)

            try:
                xm, ym = next(mem_iter)
            except StopIteration:
                mem_iter = iter(mem_loader)
                xm, ym = next(mem_iter)

            xm, ym = xm.to(device), ym.to(device)

            x = torch.cat([xb, xm], dim=0)
            y = torch.cat([yb, ym], dim=0)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

    # Final evals
    acc_A_final_taskaware = eval_taskaware(model, test_A_loader, allowed_classes=list(range(0, 50)), device=device)
    acc_B_final_taskaware = eval_taskaware(model, test_B_loader, allowed_classes=list(range(50, 100)), device=device)

    acc_A_final_cil = eval_cil(model, test_A_loader, device=device, temp=EVAL_TEMP)
    acc_B_final_cil = eval_cil(model, test_B_loader, device=device, temp=EVAL_TEMP)

    joint_loader = DataLoader(
        ConcatDataset([test_A_loader.dataset, test_B_loader.dataset]),
        batch_size=BS,
        shuffle=False,
        num_workers=NW,
        pin_memory=True
    )
    acc_joint_cil = eval_cil(model, joint_loader, device=device, temp=EVAL_TEMP)

    retention_taskaware = (acc_A_final_taskaware / max(acc_A_init, 1e-8)) * 100.0
    retention_cil = (acc_A_final_cil / max(acc_A_init, 1e-8)) * 100.0
    forgetting_taskaware = acc_A_init - acc_A_final_taskaware
    forgetting_cil = acc_A_init - acc_A_final_cil
    bwt_taskaware = acc_A_final_taskaware - acc_A_init
    bwt_cil = acc_A_final_cil - acc_A_init

    return {
        "acc_A_init": acc_A_init,
        "acc_A_final_taskaware": acc_A_final_taskaware,
        "acc_B_final_taskaware": acc_B_final_taskaware,
        "acc_A_final_cil": acc_A_final_cil,
        "acc_B_final_cil": acc_B_final_cil,
        "acc_joint_cil": acc_joint_cil,
        "retention_taskaware": retention_taskaware,
        "retention_cil": retention_cil,
        "forgetting_taskaware": forgetting_taskaware,
        "forgetting_cil": forgetting_cil,
        "bwt_taskaware": bwt_taskaware,
        "bwt_cil": bwt_cil
    }


def main():
    set_seed(0)
    os.makedirs("data", exist_ok=True)

    train_tf = transforms.Compose([
        transforms.ToTensor(),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
    ])

    train_base = CIFAR100(root="data", train=True, download=True, transform=train_tf)
    eval_base = CIFAR100(root="data", train=True, download=False, transform=eval_tf)

    targets = np.array(train_base.targets)
    all_idx = np.arange(len(targets))

    # Task-A: classes 0..49, Task-B: classes 50..99
    idx_A = all_idx[np.isin(targets, np.arange(0, 50))]
    idx_B = all_idx[np.isin(targets, np.arange(50, 100))]

    train_A_idx, val_A_idx, test_A_idx = stratified_split(idx_A, targets, val_ratio=0.2, test_ratio=0.2, seed=11)
    train_B_idx, val_B_idx, test_B_idx = stratified_split(idx_B, targets, val_ratio=0.2, test_ratio=0.2, seed=22)

    results = []
    meta = {
        "timestamp": str(datetime.utcnow()),
        "dataset": "CIFAR100",
        "method": "Strict ER baseline",
        "quick_run": QUICK_RUN,
        "epochs_A": EPOCHS_A,
        "epochs_B": EPOCHS_B,
        "replay_per_class": REPLAY_PER_CLASS,
        "eval_temp": EVAL_TEMP,
        "device": str(device),
        "seeds": SEEDS,
        "results": []
    }

    print("\n[1] Running stricter CIFAR100 ER experiments...")
    try:
        for s in SEEDS:
            print(f" -> Seed {s}...")

            memory_indices = build_memory_indices(
                train_targets=train_base.targets,
                train_A_idx=train_A_idx,
                replay_per_class=REPLAY_PER_CLASS,
                seed=s
            )

            loaders = (
                DataLoader(Subset(train_base, train_A_idx), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=(NW > 0)),
                DataLoader(Subset(eval_base, val_A_idx), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
                DataLoader(Subset(train_base, train_B_idx), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=(NW > 0)),
                DataLoader(Subset(eval_base, val_B_idx), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
                DataLoader(Subset(eval_base, test_A_idx), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
                DataLoader(Subset(eval_base, test_B_idx), batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True),
            )

            r = run_er_cifar100_strict(
                seed=s,
                loaders=loaders,
                device=device,
                train_base=train_base,
                memory_indices=memory_indices
            )

            results.append(r)
            meta["results"].append({"seed": s, **r})
            save_progress(meta)

            print(
                f"    ↳ Seed {s}: "
                f"A_init(TA)={r['acc_A_init']:.2f} | "
                f"A_final(TA)={r['acc_A_final_taskaware']:.2f} | "
                f"B_final(TA)={r['acc_B_final_taskaware']:.2f} | "
                f"A_final(CIL)={r['acc_A_final_cil']:.2f} | "
                f"B_final(CIL)={r['acc_B_final_cil']:.2f} | "
                f"Joint_CIL={r['acc_joint_cil']:.2f}"
            )

    except KeyboardInterrupt:
        print("\n⛔ Interrupted. Partial results saved to:", SAVE_PATH)
        save_progress(meta)
        return

    if results:
        print("\n" + "=" * 100)
        print("SUMMARY (Mean ± Std)")
        print("=" * 100)

        for k, label in [
            ("acc_A_init", "A Init (Task-aware)"),
            ("acc_A_final_taskaware", "A Final (Task-aware)"),
            ("acc_B_final_taskaware", "B Final (Task-aware)"),
            ("acc_A_final_cil", "A Final (CIL)"),
            ("acc_B_final_cil", "B Final (CIL)"),
            ("acc_joint_cil", "Joint Test CIL"),
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        for k, label in [
            ("retention_taskaware", "Retention (Task-aware)"),
            ("retention_cil", "Retention (CIL)"),
            ("forgetting_taskaware", "Forgetting (Task-aware)"),
            ("forgetting_cil", "Forgetting (CIL)"),
            ("bwt_taskaware", "BWT (Task-aware)"),
            ("bwt_cil", "BWT (CIL)")
        ]:
            vals = [r[k] for r in results]
            print(f"{label:26}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

        print("=" * 100)
        print(f"Saved run log to: {SAVE_PATH}")


if __name__ == "__main__":
    main()

100%|██████████| 169M/169M [00:04<00:00, 35.2MB/s] 



[1] Running stricter CIFAR100 ER experiments...
 -> Seed 1...


/tmp/ipykernel_55/2894767124.py:283: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": str(datetime.utcnow()),


    ↳ Seed 1: A_init(TA)=41.76 | A_final(TA)=39.34 | B_final(TA)=43.18 | A_final(CIL)=26.28 | B_final(CIL)=37.50 | Joint_CIL=31.89
 -> Seed 2...
    ↳ Seed 2: A_init(TA)=44.54 | A_final(TA)=40.10 | B_final(TA)=43.76 | A_final(CIL)=27.26 | B_final(CIL)=38.00 | Joint_CIL=32.63
 -> Seed 3...
    ↳ Seed 3: A_init(TA)=43.32 | A_final(TA)=40.44 | B_final(TA)=44.14 | A_final(CIL)=28.64 | B_final(CIL)=38.74 | Joint_CIL=33.69

SUMMARY (Mean ± Std)
A Init (Task-aware)       : 43.21 ± 1.14
A Final (Task-aware)      : 39.96 ± 0.46
B Final (Task-aware)      : 43.69 ± 0.39
A Final (CIL)             : 27.39 ± 0.97
B Final (CIL)             : 38.08 ± 0.51
Joint Test CIL            : 32.74 ± 0.74
Retention (Task-aware)    : 92.53 ± 1.80
Retention (CIL)           : 63.42 ± 2.03
Forgetting (Task-aware)   : 3.25 ± 0.86
Forgetting (CIL)          : 15.81 ± 1.09
BWT (Task-aware)          : -3.25 ± 0.86
BWT (CIL)                 : -15.81 ± 1.09
Saved run log to: strict_er_cifar100_results.json
